# Multi-year hydrogen pathways and capture credits

Plan equipment additions for rising hydrogen demand. Compare renewable electrolysis, steam methane reforming with carbon capture, and a portfolio allowed to invest in both. The model carries installed capacity forward from one planning year to the next and charges for new capacity only.

The capacity factors and cost projections are illustrative. The reforming gas requirement, residual emissions, and captured CO₂ coefficients come from the legacy hydrogen pathways case. The credit values are scenario assumptions.

## Planning inputs

Each year represents 8,760 hours at the average conditions shown. Costs are undiscounted across the three sampled planning years. The emissions figures cover on-site vented CO₂ during operation.

In [1]:
from math import isclose

import pandas as pd
from energia import Model, Periods, Process, Resource
from gana import inf

years = [2025, 2030, 2035]
demand = [0.2, 0.3, 0.45]  # tonnes of H₂ per hour
gas_price = [175, 200, 235]  # currency per tonne of natural gas
credit = [20.22, 20.22, 35]  # currency per tonne of sequestered CO₂
pv_availability = [0.25, 0.27, 0.29]
hours_per_year = 8760

## Build a pathway model

A common hydrogen balance lets the model split production between routes. In this example captured CO₂ can be sequestered up to the capture plant's capacity. The investment term is the cost of capacity additions, including the initial installation.

In [2]:
def plan_hydrogen(pathway, capture_credit):
    m = Model(pathway)
    m.year = Periods()
    m.study = len(years) * m.year
    m.declare(
        Resource,
        ["solar", "gas", "power", "hydrogen", "vented_co2", "captured_co2", "sequestered_co2"],
    )
    for resource in (m.power, m.hydrogen, m.vented_co2, m.captured_co2, m.sequestered_co2):
        _ = resource.consume(m.year) == 0
    for resource in (m.solar, m.gas):
        _ = resource.consume(m.year) >= 0
    served = m.hydrogen.release(m.year)
    _ = served == demand
    _ = m.vented_co2.release(m.year) >= 0
    _ = m.sequestered_co2.release(m.year) >= 0

    assets = {}
    if pathway in ("Green", "Portfolio"):
        assets.update({
            "pv": (m.power == -m.solar, pv_availability, [850_000, 680_000, 550_000]),
            "electrolyzer": (m.hydrogen == -52 * m.power, 1, [2_000_000, 1_400_000, 950_000]),
        })
    if pathway in ("Blue", "Portfolio"):
        assets.update({
            "reformer": (
                m.hydrogen == -3.76 * m.gas + 1.03 * m.vented_co2 + 9.332 * m.captured_co2,
                1, [2_520_000, 2_450_000, 2_375_000],
            ),
            "sequestration": (
                m.sequestered_co2 == -m.captured_co2,
                1, [552_000, 520_000, 500_000],
            ),
        })

    capex = 0
    for name, (conversion, availability, costs) in assets.items():
        plant = Process(conversion)
        setattr(m, name, plant)
        capacity = plant.capacity(m.year)
        _ = capacity <= 200
        _ = plant.operate(m.year) <= availability
        plant.locate()
        installed = list(capacity.V())
        additions = [installed[0], *(later - previous for previous, later in zip(installed, installed[1:]))]
        for year, addition in zip(years[1:], additions[1:], strict=True):
            setattr(m.program, f"{name}_addition_{year}", addition >= 0)
        capex += sum(cost * amount for cost, amount in zip(costs, additions, strict=True))

    annual_gas = m.gas.consume(m.year).V()
    annual_sequestration = m.sequestered_co2.release(m.year).V()
    fuel_cost = sum(price * gas for price, gas in zip(gas_price, annual_gas, strict=True))
    avoided_cost = sum(rate * flow for rate, flow in zip(capture_credit, annual_sequestration, strict=True))
    m.program.total_cost = inf(capex + hours_per_year * (fuel_cost - avoided_cost))
    m.opt()

    assert isclose(sum(served.output(aslist=True)), sum(demand), abs_tol=1e-6)
    vented = m.vented_co2.release(m.year).output(aslist=True)
    return {
        "Planning cost": m.program.active_objective.X,
        "Vented CO₂ (tonnes)": hours_per_year * sum(vented),
    }

## Compare investments

In [3]:
zero_credit = [0] * len(years)
results = pd.DataFrame({
    "Green only": plan_hydrogen("Green", zero_credit),
    "Blue only": plan_hydrogen("Blue", zero_credit),
    "Portfolio without credit": plan_hydrogen("Portfolio", zero_credit),
    "Portfolio with credit": plan_hydrogen("Portfolio", credit),
}).T
assert results.loc["Green only", "Vented CO₂ (tonnes)"] == 0
assert results.loc["Blue only", "Vented CO₂ (tonnes)"] > 0
assert results.loc["Portfolio with credit", "Planning cost"] <= results.loc["Portfolio without credit", "Planning cost"] + 1e-6
results.round(2)

,Planning cost,Vented CO₂ (tonnes)
Green only,59644921.46,0.00
Blue only,9932890.00,8571.66
Portfolio without credit,9932890.00,8571.66
Portfolio with credit,7818878.44,8571.66
